# LLMOps: Open Source Models & HuggingFace

In [ ]:
# Day 48 — Open Source LLMs & HuggingFace
#
# The OpenAI API is fantastic, but it has downsides:
# 1. Cost: Sending millions of tokens gets very expensive.
# 2. Privacy: Some enterprise data (like PII or healthcare data) CANNOT be sent to a 3rd party API.
# 3. Vendor Lock-in: If OpenAI goes down, your app goes down.
#
# The alternative is Open-Source Models (like Meta's LLaMA 3, Mistral, or Google's Gemma).
# You download the model weights to your own server and run them locally.
# HuggingFace is the "GitHub of Machine Learning" where these models live.
#
# Today we'll cover:
#   1. The Transformers Library Pipeline
#   2. Running local LLMs (CPU vs GPU)
#   3. Connecting HuggingFace to LangChain
#
# Install: pip install transformers torch

### 1. The Transformers Pipeline

In [ ]:
# The easiest way to use HuggingFace models is the `pipeline` abstraction.
# Note: Running a full LLM takes a massive GPU. For this demo, we use a tiny model 
# (like a sentiment analyzer or a tiny text generator) that runs on CPU.

hf_pipeline_code = '''from transformers import pipeline

# 1. Initialize a pipeline for text classification (sentiment analysis)
# By default, it downloads a small model (distilbert-base-uncased-finetuned-sst-2-english)
print("Downloading/Loading Model...")
sentiment_classifier = pipeline("sentiment-analysis")

# 2. Run inference locally (No API call!)
result1 = sentiment_classifier("The new Kubernetes deployment was incredibly smooth and fast!")
result2 = sentiment_classifier("The database crashed again, this infrastructure is terrible.")

print(f"Result 1: {result1}") # [{'label': 'POSITIVE', 'score': 0.99}]
print(f"Result 2: {result2}") # [{'label': 'NEGATIVE', 'score': 0.99}]
'''

print("HuggingFace Pipeline Pattern:")
print(hf_pipeline_code)

### 2. Loading a Text Generation LLM

In [ ]:
# To load a conversational LLM (like GPT-2 or a tiny Llama variant),
# we specify the model ID from huggingface.co

hf_llm_code = '''from transformers import pipeline

# GPT-2 is very old and small, but it demonstrates text generation locally.
# Modern open models (Llama 3 8B) require about 16GB of VRAM.
generator = pipeline('text-generation', model='gpt2')

prompt = "In modern DevOps, the most important tool is"
response = generator(prompt, max_length=50, num_return_sequences=1)

print(response[0]['generated_text'])
'''

print("Local Text Generation Pattern:")
print(hf_llm_code)

### 3. Integrating HuggingFace with LangChain

In [ ]:
# Remember LCEL and Agents from Day 46? 
# LangChain's genius is that you can swap OpenAI for HuggingFace seamlessly.

hf_langchain_code = '''from langchain_community.llms.huggingface_pipeline import HuggingFacePipeline
from langchain_core.prompts import PromptTemplate

# Create the local pipeline
hf_pipe = pipeline("text-generation", model="gpt2", max_new_tokens=50)

# Wrap it in a LangChain LLM object!
local_llm = HuggingFacePipeline(pipeline=hf_pipe)

# Now you can use it exactly like ChatOpenAI()
template = "What is {technology}?"
prompt = PromptTemplate.from_template(template)

chain = prompt | local_llm

print(chain.invoke({"technology": "Docker"}))
'''

print("Swapping APIs for Local Models in LangChain:")
print(hf_langchain_code)

In [ ]:
# Practice Questions:
#
# 1. What are the three main reasons a company might choose an Open-Source HuggingFace model 
#    over the OpenAI API?
# 2. Why can't you easily run Meta's Llama 3 (70 Billion parameters) on a standard laptop CPU?
# 3. How does LangChain make it easy to prevent Vendor Lock-in with AI models?

### DevOps / AI-MLOps Practice

In [ ]:
# Exercise 1: The Secure Log Analyzer
# You need to analyze Nginx access logs to find malicious requests.
# These logs contain customer IP addresses and session cookies (highly sensitive).
# 
# Write pseudo-code using HuggingFacePipeline and LangChain to load a local 
# Mistral-7B model, pass it a batch of log lines, and output JSON classifications 
# of "Safe" or "Malicious". Ensure no data leaves the local network.

# Why this matters for me as a DevOps/MLOps engineer:
# 1. Data Privacy: Data privacy regulations (GDPR/HIPAA) often forbid sending data to OpenAI. 
#    Running open models inside your VPC is the only legal way to use LLMs in these industries.
# 2. MLOps Infrastructure: Deploying HuggingFace models to production requires heavy DevOps work 
#    (Kubernetes, GPUs, vLLM/TGI inference servers). Knowing how the code works is step 1.